# GCP Pose Estimation — Kaggle Training
GPU + Internet must be enabled. This is the safety gate: 3-epoch smoke first, eyeball predictions, then full run.

In [ ]:
# 1. Code + deps
!git clone <YOUR_REPO_URL> gcp
%cd gcp
!pip -q install -r requirements.txt

In [ ]:
# 2. Data — download Drive folder and normalise paths
# gdown --folder creates data/<drive-folder-name>/ as a subdirectory;
# this cell finds it and rewires configs/config.yaml automatically.
import os, shutil, yaml

os.makedirs('data', exist_ok=True)
os.system('gdown --folder 1RDNiAO1EynKrRDomcVNXQW1-ct5zzvE5 -O data/ --remaining-ok')

# Locate the downloaded subfolder (Drive names the dir after the folder title)
candidates = [d for d in os.listdir('data')
              if os.path.isdir(f'data/{d}') and d not in ('train_dataset', 'test_dataset')]
if candidates:
    # Move contents up one level so data/ is the root
    src = f'data/{candidates[0]}'
    for item in os.listdir(src):
        dst = f'data/{item}'
        if not os.path.exists(dst):
            shutil.move(f'{src}/{item}', dst)
    if not os.listdir(src):
        os.rmdir(src)
    print(f'Moved contents of {src} -> data/')

print('data/ contents:', sorted(os.listdir('data')))
assert os.path.exists('data/gcp_marks.json'), 'gcp_marks.json not found under data/'
assert os.path.isdir('data/train_dataset'), 'train_dataset/ not found under data/'
assert os.path.isdir('data/test_dataset'),  'test_dataset/ not found under data/'

# Confirm config paths (they already match after the move)
cfg = yaml.safe_load(open('configs/config.yaml'))
cfg['paths'] = {'label_file': 'data/gcp_marks.json',
                'train_dir':  'data/train_dataset',
                'test_dir':   'data/test_dataset',
                'output_dir': 'outputs'}
yaml.safe_dump(cfg, open('configs/config.yaml', 'w'))
print('Config paths verified.')

In [ ]:
# 3. SMOKE: 3 epochs, then inspect predictions schema before committing GPU time
import yaml
c = yaml.safe_load(open('configs/config.yaml')); c['training']['num_epochs'] = 3
yaml.safe_dump(c, open('configs/config.yaml', 'w'))
!python train.py --config configs/config.yaml
!python predict.py --config configs/config.yaml --checkpoint outputs/best.pt --output predictions.json
import json; p = json.load(open('predictions.json'))
k = next(iter(p)); print(k, '->', p[k]); print('total:', len(p))

In [ ]:
# 4. FULL run: restore 40 epochs
c = yaml.safe_load(open('configs/config.yaml')); c['training']['num_epochs'] = 40
yaml.safe_dump(c, open('configs/config.yaml', 'w'))
!python train.py --config configs/config.yaml

In [ ]:
# 5. Final predictions + download artifacts
!python predict.py --config configs/config.yaml --checkpoint outputs/best.pt --output predictions.json
import pandas as pd; pd.read_csv('outputs/training_log.csv').tail()

Download `outputs/best.pt`, `outputs/training_log.csv`, and `predictions.json`. Put `best.pt` on Drive and paste the link + the final PCK/F1 into `README.md`.